In [1]:

def _features_lower(listing):
    # Use an empty list if the features field is missing or None.
    # casefold() allows feature names to be compared without case differences.
    return {f.lower() for f in (listing.get("features") or [])}


def filter_listings(listings, requirements):
    """Return listings that satisfy every specified hard constraint."""
    # Get the preferred location.
    # Remove surrounding spaces and make matching case-insensitive.
    # An empty string means that any location is acceptable.
    location = requirements.get("location")
    location = location.strip().casefold() if isinstance(location, str) else ""
    # Get the user's numeric limits.
    # A value of None means that the constraint was not specified.
    max_rent = requirements.get("max_rent_pw")
    min_bedrooms = requirements.get("min_bedrooms")
    max_distance = requirements.get("max_distance_km")
    # Get the required features.
    # Use an empty list when no must-have features were specified.
    must_have = requirements.get("must_have_features") or []
    # Convert required features to a case-insensitive set.
    # A set makes it easy to check whether all required features are present.
    must_have_lower = {feature.casefold() for feature in must_have}

    filtered = []
    # Check each property listing individually.
    for listing in listings:
        # Extract the fields needed for filtering.
        rent = listing.get("weekly_rent")
        bedrooms = listing.get("bedrooms")
        suburb = listing.get("suburb")
        distance = listing.get("distance_to_uni_km")
        # Convert the listing's features to a case-insensitive set.
        listing_features = _features_lower(listing)

        # Reject the listing if a location was provided and the suburb
        # does not contain that location keyword.
        if location and location not in suburb.casefold():
            continue
        # Reject the listing if it exceeds the maximum weekly rent.
        if max_rent is not None and rent > max_rent:
            continue
        # Reject the listing if it has fewer than the required bedrooms.
        # A minimum of zero means that there is no bedroom constraint.
        if min_bedrooms is not None and min_bedrooms > 0 and bedrooms < min_bedrooms:
            continue
        # Reject the listing if it is farther away than allowed.
        if max_distance is not None and distance > max_distance:
            continue
        # Reject the listing unless every must-have feature is present.
        if not must_have_lower.issubset(listing_features):
            continue
        # The listing passed every constraint, so keep it.
        filtered.append(listing)
    # Return all listings that satisfy the user's requirements.
    return filtered


In [2]:
def score_listing(listing, requirements):
    """Calculate a preference score; a higher score is better.
    Score formula = 0.5 * feature_score + 0.3 * rent_score + 0.2 * distance_score
    feature_score = len(matched) / len(nice_lower)
    rent_score = (max_rent - rent) / max_rent
    distance_score = (max_distance - distance) / max_distance
    """

    nice_to_have = requirements.get("nice_to_have_features") or []
    nice_lower = {feature.casefold() for feature in nice_to_have}
    if not nice_lower:
        feature_score = 0.0
    else:
        listing_features = _features_lower(listing)
        matched = nice_lower & listing_features
        feature_score = len(matched) / len(nice_lower)

    max_rent = requirements.get("max_rent_pw")
    rent = listing.get("weekly_rent")
    if not max_rent or rent is None:
        rent_score = 0.0
    else:
        rent_score = max(0.0, (max_rent - rent) / max_rent)

    max_distance = requirements.get("max_distance_km")
    distance = listing.get("distance_to_uni_km")
    if not max_distance or distance is None:
        distance_score = 0.0
    else:
        distance_score = max(0.0, (max_distance - distance) / max_distance)


    score = 0.5 * feature_score + 0.3 * rent_score + 0.2 * distance_score
    return round(score, 3)

In [3]:
def score_listing(listing, requirements):
    """Score how well a listing matches the user's soft preferences.

    Returns a float from 0.0 to 1.0; higher is better.

        score = 0.5 * feature_score + 0.3 * rent_score + 0.2 * distance_score

    Each component is normalised to 0-1 first, because the raw values sit on
    very different scales (0-3 features, $360-875 rent, 0.5-8.2 km) and would
    otherwise let rent dominate the total.

        feature_score  = matched nice-to-haves / total nice-to-haves
        rent_score     = (max_rent - rent) / max_rent           lower rent scores higher
        distance_score = (max_distance - distance) / max_distance

    Weighting rationale: nice-to-have features carry the most weight because
    the user named them explicitly. Rent outweighs distance because it is a
    recurring weekly cost, while distance is a one-off inconvenience per trip.
    Weights sum to 1.0 so the score stays interpretable as a percentage match.

    Any preference the user left unspecified contributes 0.0 rather than
    raising, so the function is safe on a partly filled requirements dict.
    """
    # Nice-to-have features: proportion of requested features present
    nice_to_have = requirements.get("nice_to_have_features") or []
    # casefold both sides so "Balcony" and "balcony" match
    nice_lower = {feature.casefold() for feature in nice_to_have}

    if not nice_lower:
        feature_score = 0.0          # none requested; also avoids divide-by-zero
    else:
        listing_features = _features_lower(listing)
        matched = nice_lower & listing_features      # set intersection
        feature_score = len(matched) / len(nice_lower)

    # Rent: cheaper is better, measured against the user's budget
    max_rent = requirements.get("max_rent_pw")
    rent = listing.get("weekly_rent")
    if not max_rent or rent is None:
        # No budget stated (or a nonsensical 0), so rent cannot influence rank.
        rent_score = 0.0
    else:
        # max() floors the result: an over-budget listing scores 0, not negative.
        rent_score = max(0.0, (max_rent - rent) / max_rent)

    # Distance: closer to the university is better
    max_distance = requirements.get("max_distance_km")
    distance = listing.get("distance_to_uni_km")
    if not max_distance or distance is None:
        distance_score = 0.0
    else:
        distance_score = max(0.0, (max_distance - distance) / max_distance)

    score = 0.5 * feature_score + 0.3 * rent_score + 0.2 * distance_score
    # 3 dp rather than 2: fewer artificial ties when top_n sorts on this value.
    return round(score, 3)

In [4]:
def top_n(listings, requirements, n=3):
    """Return the n best-matching listings, highest score first."""
    must_have = requirements.get("must_have_features") or []
    must_have_lower = {feature.casefold() for feature in must_have}
    nice_to_have = requirements.get("nice_to_have_features") or []
    nice_lower = {feature.casefold() for feature in nice_to_have}

    results = []
    for listing in filter_listings(listings, requirements):
        listing_features = _features_lower(listing)

        result = dict(listing)
        result["score"] = score_listing(listing, requirements)
        result["matched_must_haves"] = sorted(must_have_lower & listing_features)
        result["matched_nice_to_haves"] = sorted(nice_lower & listing_features)
        results.append(result)
    # sort by score descending; if score are tied then use id ascending to break ties
    results.sort(key=lambda r: (-r["score"], r["id"]))
    return results[:n]



In [5]:

def top_n(listings, requirements, n=3):
    """Return the n best-matching listings, highest score first."""

    # Get the user's required features.
    # Use an empty list if the field is missing or None.
    must_have = requirements.get("must_have_features") or []

    # Convert required features to a case-insensitive set.
    must_have_lower = {
        feature.casefold()
        for feature in must_have
    }

    # Get the user's preferred—but not required—features.
    nice_to_have = requirements.get("nice_to_have_features") or []

    # Convert preferred features to a case-insensitive set.
    nice_lower = {
        feature.casefold()
        for feature in nice_to_have
    }

    # Store the scored recommendation records.
    results = []

    # First apply all hard constraints using filter_listings().
    for listing in filter_listings(listings, requirements):
        # Convert the listing's features to a case-insensitive set.
        listing_features = _features_lower(listing)

        # Make a shallow copy so adding recommendation fields does not
        # modify the original listing dictionary.
        result = dict(listing)

        # Calculate and add the listing's recommendation score.
        result["score"] = score_listing(listing, requirements)

        # Find the required features that the listing contains.
        # The & operator calculates the intersection of two sets.
        result["matched_must_haves"] = sorted(
            must_have_lower & listing_features
        )

        # Find the preferred features that the listing contains.
        result["matched_nice_to_haves"] = sorted(
            nice_lower & listing_features
        )

        # Add the completed recommendation record to the results.
        results.append(result)

    # Sort recommendations by:
    # 1. Score from highest to lowest
    # 2. Listing ID from lowest to highest when scores are tied
    #
    # Scores are negated because Python normally sorts numbers in
    # ascending order. For example, -10 comes before -5, meaning a
    # score of 10 is placed before a score of 5.
    results.sort(
        key=lambda result: (
            -result["score"],
            result["id"],
        )
    )

    # Return only the first n recommendations.
    return results[:n]

In [6]:
recommendations = top_n(listings, REQUIREMENTS, n=3)
print(f"Found {len(recommendations)} recommendation(s):\n")
for r in recommendations:
    print(
        f"#{r['id']} {r['suburb']} ${r['rent_pw']}/wk  {r['bedrooms']}br  score={r['score']}")

NameError: name 'listings' is not defined